In [2]:
!pip install Bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.4/321.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00


In [9]:
!wget https://github.com/shenwei356/seqkit/releases/download/v2.5.1/seqkit_linux_amd64.tar.gz
!tar -xzf seqkit_linux_amd64.tar.gz
!chmod +x seqkit
!sudo mv seqkit /usr/local/bin/
!seqkit version

--2026-06-17 12:38:21--  https://github.com/shenwei356/seqkit/releases/download/v2.5.1/seqkit_linux_amd64.tar.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/52715040/e5a1bc13-90fd-42ce-b9e1-8b482d8f8b57?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-17T13%3A18%3A14Z&rscd=attachment%3B+filename%3Dseqkit_linux_amd64.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-17T12%3A18%3A06Z&ske=2026-06-17T13%3A18%3A14Z&sks=b&skv=2018-11-09&sig=mdJWh%2FMt2BN0Ne45xXymevy9aGaFQB06BL5P8AtO0CA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MTcwMDEzOSwibmJmIjoxNzgxNjk5ODM5LCJwYXRoIjoicmVsZWFzZWFzc2V

In [10]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

In [11]:
class MyFastaParser:
    """
    Class for parsing FASTA files with seqkit calls and database queries.
    """
    def __init__(self, file_name):
        """
        Initializes the parser with the specified file name.

        Args:
            file_name (str): Path to the FASTA file
        """
        self.filename = file_name
        # Regular expressions for finding IDs in sequence descriptions
        self.uniprot_pattern = re.compile(
            r"(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})(?:-\d+)?"
        )
        self.ensembl_pattern = re.compile(
            r"ENS[A-Z]{0,10}(?:E|FM|G|GT|P|R|T)\d{6,}(?:\.\d+)?"
        )

    def _get_uniprot(self, accession):
        """
        Internal method for querying the UniProt API.

        Args:
            accession (str): UniProt accession identifier

        Returns:
            requests.Response: Response from the API
        """
        endpoint = "https://rest.uniprot.org/uniprotkb/accessions"
        http_function = requests.get
        http_args = {"params": {"accessions": accession}, "timeout": 30}
        return http_function(endpoint, **http_args)

    def _get_ensembl(self, id):
        """
        Internal method for querying the ENSEMBL API.

        Args:
            id (str): ENSEMBL identifier

        Returns:
            requests.Response: Response from the API
        """
        endpoint = f"https://rest.ensembl.org/lookup/id/{id.split('.')[0]}"
        http_function = requests.get
        http_args = {"headers": {"Content-Type": "application/json"}, "timeout": 30}
        return http_function(endpoint, **http_args)

    def _uniprot_parse_response(self, resp: dict):
        """
        Parses the response from the UniProt API.

        Args:
            resp (dict): JSON response from UniProt

        Returns:
            dict: Structured information about the protein
        """
        resp = resp.json()
        output = {}
        for val in resp["results"]:
            accession = val["primaryAccession"]
            output[accession] = {
                "organism": val["organism"]["scientificName"],
                "geneInfo": val["genes"],
                "sequenceInfo": val["sequence"],
                "type": "protein"
            }
        return output

    def _ensembl_parse_response(self, resp: dict):
        """
        Parses the response from the ENSEMBL API.

        Args:
            resp (dict): JSON response from ENSEMBL

        Returns:
            dict: Structured information about the gene/transcript
        """
        resp = resp.json()
        id = resp["id"]
        output = {
            id: {
                "object_type": resp.get("object_type"),
                "assembly_name": resp.get("assembly_name"),
                "species": resp.get("species"),
                "db_type": resp.get("db_type"),
                "biotype": resp.get("biotype"),
                "display_name": resp.get("display_name"),
                "id": resp.get("id"),
                "description": resp.get("description"),
                "canonical_transcript": resp.get("canonical_transcript"),
                "source": resp.get("source")
            }
        }
        return output

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        """
        Queries the appropriate database and parses the response.

        Args:
            id (str): Identifier to search for
            database (str): 'uniprot' or 'ensembl'
            seq_description (str): Sequence description from FASTA
            seq_sequence (str): The sequence itself

        Returns:
            dict: Combined information from FASTA and the database
        """
        output = {
            f"file_info_{id}": {
                "description": seq_description,
                "sequence": seq_sequence
            }
        }
        response = None
        try:
            if database == "uniprot":
                response = self._get_uniprot(id)
                response.raise_for_status()
                parsed = self._uniprot_parse_response(response)
            else:
                response = self._get_ensembl(id)
                response.raise_for_status()
                parsed = self._ensembl_parse_response(response)
            output[f"database_info_{id}"] = next(iter(parsed.values()))
        except requests.RequestException as error:
            if response is not None:
                try:
                    error_output = response.json()
                except ValueError:
                    error_output = str(error)
            else:
                error_output = str(error)
            output[f"database_info_{id}"] = error_output
        return output

    def seqkit_stats(self) -> dict:
        """
        Calls seqkit stats to obtain statistics for the FASTA file.

        Returns:
            dict: File statistics or error
        """
        try:
            seqkit = subprocess.run(
                ("seqkit", "stats", self.filename, "-a"),
                capture_output=True,
                text=True
            )
        except FileNotFoundError as error:
            return {"error": str(error)}
        if seqkit.stderr:
            return {"error": seqkit.stderr.strip()}
        seqkit_out = seqkit.stdout.strip().split("\n")
        prop_names = seqkit_out[0].split()[1:]
        prop_vals = seqkit_out[1].split()[1:]
        seq_info = dict(zip(prop_names, prop_vals))
        seqkit_result = {
            "fasta_seqkit_stat_info": seq_info,
            "fasta_type": seq_info["type"],
            "fasta_num_seqs": int(seq_info["num_seqs"].replace(",", ""))
        }
        return seqkit_result

    def biopython_parser(self, seqkit_result) -> dict:
        """
        Parses the FASTA file using Biopython and queries databases.

        Args:
            seqkit_result (dict): Result from seqkit_stats()

        Returns:
            dict: Information about each sequence with database data
        """
        if "fasta_type" not in seqkit_result:
            return seqkit_result
        if seqkit_result["fasta_type"] == "Protein":
            pattern = self.uniprot_pattern
            database = "uniprot"
        else:
            pattern = self.ensembl_pattern
            database = "ensembl"
        output = {"DB_name": database}
        warning = False
        for seq in SeqIO.parse(self.filename, "fasta"):
            match = pattern.search(seq.description)
            if match is None:
                warning = True
                continue
            output.update(
                self._access_database(
                    match.group(0),
                    database,
                    seq.description,
                    str(seq.seq)
                )
            )
        if warning:
            output["WARNING"] = {"No ID match found."}
        return output

    def show_output(self, output, indent=0):
        """
        Pretty-prints structured data.

        Args:
            output (dict): Data to print
            indent (int): Indentation level
        """
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [12]:
# Testing on the test file
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
print("=== Statistics seqkit for test_file.fasta ===")
print(json.dumps(stats, indent=2, ensure_ascii=False))

=== Statistics seqkit for test_file.fasta ===
{
  "fasta_seqkit_stat_info": {
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "2",
    "sum_len": "456",
    "min_len": "29",
    "avg_len": "228",
    "max_len": "427",
    "Q1": "29",
    "Q2": "228",
    "Q3": "427",
    "sum_gap": "0",
    "N50": "427",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "GC(%)": "8.33"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 2
}


In [14]:
biopython = parser.biopython_parser(stats)
print("\n=== Parsing results for test_file.fasta ===")
parser.show_output(biopython)


=== Parsing results for test_file.fasta ===
DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSD

In [15]:
# Testing on the file with UniProt data
parser_uniprot = MyFastaParser('uniprot_download.fasta')
stats_uniprot = parser_uniprot.seqkit_stats()
print("=== seqkit stats for uniprot_download.fasta ===")
print(json.dumps(stats_uniprot, indent=2, ensure_ascii=False))

biopython_uniprot = parser_uniprot.biopython_parser(stats_uniprot)
print("\n=== Parsing results for uniprot_download.fasta ===")
parser_uniprot.show_output(biopython_uniprot)

=== seqkit stats for uniprot_download.fasta ===
{
  "fasta_seqkit_stat_info": {
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "7",
    "sum_len": "3,861",
    "min_len": "180",
    "avg_len": "551.6",
    "max_len": "1,382",
    "Q1": "429",
    "Q2": "441",
    "Q3": "500",
    "sum_gap": "0",
    "N50": "468",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "GC(%)": "9.14"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 7
}

=== Parsing results for uniprot_download.fasta ===
DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLK

In [16]:
# Testing on the 1st ENSEMBL file
parser_ensembl1 = MyFastaParser('ensembl_download_1.fasta')
stats_ensembl1 = parser_ensembl1.seqkit_stats()
print("=== seqkit stats for ensembl_download_1.fasta ===")
print(json.dumps(stats_ensembl1, indent=2, ensure_ascii=False))

biopython_ensembl1 = parser_ensembl1.biopython_parser(stats_ensembl1)
print("\n=== Parsing results for ensembl_download_1.fasta ===")
parser_ensembl1.show_output(biopython_ensembl1)

=== seqkit stats for ensembl_download_1.fasta ===
{
  "fasta_seqkit_stat_info": {
    "format": "FASTA",
    "type": "DNA",
    "num_seqs": "6",
    "sum_len": "86",
    "min_len": "9",
    "avg_len": "14.3",
    "max_len": "23",
    "Q1": "10",
    "Q2": "13.5",
    "Q3": "17",
    "sum_gap": "0",
    "N50": "16",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "GC(%)": "45.35"
  },
  "fasta_type": "DNA",
  "fasta_num_seqs": 6
}

=== Parsing results for ensembl_download_1.fasta ===
DB_name
	ensembl
file_info_ENSMUST00000196221.2
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221.2
	object_type
		Transcript
	assembly_name
		GRCm39
	species
		mus_musculus
	db_type
		core
	biotype
		TR_D_gene
	display_name
		Trdd1-202
	id
		ENSMUST0000

In [17]:
# Testing on the 2nd ENSEMBL file
parser_ensembl2 = MyFastaParser('ensembl_download_2.fasta')
stats_ensembl2 = parser_ensembl2.seqkit_stats()
print("=== seqkit stats for ensembl_download_2.fasta ===")
print(json.dumps(stats_ensembl2, indent=2, ensure_ascii=False))

biopython_ensembl2 = parser_ensembl2.biopython_parser(stats_ensembl2)
print("\n=== Parsing results for ensembl_download_2.fasta ===")
parser_ensembl2.show_output(biopython_ensembl2)

=== seqkit stats for ensembl_download_2.fasta ===
{
  "error": "\u001b[ERRO]\u001b ensembl_download_2.fasta: fastx: invalid FASTA/Q format"
}

=== Parsing results for ensembl_download_2.fasta ===
error
	[ERRO] ensembl_download_2.fasta: fastx: invalid FASTA/Q format
